In [1]:
# Identifiers (RQx, Hx, Px, Gx, Fx, Tx) are defined in vpir_bench_spec.md: section 2 for the research questions,
# section 11 for the tests behind each gate, section 12 for phases, gates and figures. Rx reproduction targets are
# in docs/reproduction_targets.md, and numbered decisions in plan.md.
# RQ6 (concurrent clients) and RQ7 (server cores) at 128 MiB x 1 KiB (decision 25), 3 load-generator runs per
# point, 30 s windows (decision 28)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import amortise, frame, plots
from cs4406_sem_project.harness import loadgen, session_check
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


plots._style()
CELL = (128 << 20, 1 << 10)  # decision 28: the RQ4 cell, 128 MiB x 1 KiB (S3 at 1 bit)

LOADRUN = repo / "results/raw/p9-20261113/loadgen"
lg = frame.load_loadgen(LOADRUN, platform)
t = plots.build(df, status, loadgen_rows=lg)
load = loadgen.yaml.safe_load(loadgen.LOAD_PATH.read_text())
print(f"{len(lg)} load-generator runs over {lg.cell_id.nunique()} cells;", lg.groupby("experiment").size().to_dict())

273 load-generator runs over 7 cells; {'closed': 84, 'multicore': 84, 'open': 105}


In [2]:
# Gate G7 (spec 11, test_loadgen_not_bottleneck): the load generator's CPU stays under the limit at the highest C.
rows = loadgen.read_rows(LOADRUN)
problems = loadgen.g7_problems(rows, load["loadgen_cpu_limit_pct"])
print("\n".join(problems) or f"G7 PASS: load generator under {load['loadgen_cpu_limit_pct']} % CPU at every top C")
print("load-generator CPU %, max per arm:", pd.DataFrame(rows).groupby("arm").loadgen_cpu_util.max().round(0).to_dict())

G7 PASS: load generator under 70 % CPU at every top C
load-generator CPU %, max per arm: {'M0': 15.0, 'M1': 14.0, 'S0-cpp': 10.0, 'S0-go': 17.0, 'S1': 11.0, 'S2': 8.0, 'S3': 1.0}


In [3]:
# RQ6: the closed-loop curve per arm (median of 3 runs per C; the sweep stops once p95 passes 10 x the C = 1 p95),
# the saturation point (highest closed-loop throughput) and, per price pair, baseline over verifiable throughput
# at saturation. H6 expects that ratio to stay near the single-query server-time ratio.
cur = t.load_curves
closed = cur[cur.experiment == "closed"].sort_values(["arm", "clients"])
print(closed[["arm", "clients", "throughput_qps", "lat_p50_s", "lat_p95_s", "server_cpu_util", "saturated_runs", "n"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
sat = plots.saturation(cur).set_index("arm")
print("\nsaturation:\n", sat[["clients", "throughput_qps", "lat_p50_s", "lat_p95_s"]].to_string(float_format=lambda v: f"{v:.4g}"))
single = t.medians[(t.medians.db_bytes == CELL[0]) & (t.medians.record_bytes == CELL[1])].set_index("arm")
for p in plots.PRICE_PAIRS:
    if p.num in sat.index and p.den in sat.index:
        print(f"{p.label}: saturation throughput ratio (baseline / verifiable) "
              f"{sat.throughput_qps[p.den] / sat.throughput_qps[p.num]:.3f}; single-query server_online_s ratio "
              f"(verifiable / baseline) {single.server_online_s[p.num] / single.server_online_s[p.den]:.3f}")

   arm  clients  throughput_qps  lat_p50_s  lat_p95_s  server_cpu_util  saturated_runs  n
    M0        1            39.1    0.02526    0.02702            195.1               0  3
    M0        4           124.6    0.03175    0.03876            723.4               0  3
    M0       16           127.1      0.124     0.1649            752.3               0  3
    M0       64           123.7     0.5086      0.582            730.4               3  3
    M1        1           27.27    0.03646    0.03868            197.3               0  3
    M1        4            86.9    0.04504    0.05481            729.4               0  3
    M1       16            88.6       0.18     0.2232            756.3               0  3
    M1       64           88.43     0.7221     0.7716            755.5               3  3
S0-cpp        1           65.53    0.01527     0.0154            98.61               0  3
S0-cpp        4           257.6    0.01549    0.01562            393.7               0  3
S0-cpp    

In [4]:
# RQ6, open loop: p50 and p95 latency per offered load (fractions of each arm's saturation throughput).
op = cur[cur.experiment == "open"].sort_values(["arm", "offered_load_qps"])
print(op[["arm", "point", "offered_load_qps", "throughput_qps", "lat_p50_s", "lat_p95_s", "saturated_runs", "n"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

   arm         point  offered_load_qps  throughput_qps  lat_p50_s  lat_p95_s  saturated_runs  n
    M0 open_w6_f0.25             31.78           32.93    0.02613    0.03172               0  3
    M0 open_w6_f0.50             63.57           64.63    0.02654    0.04322               0  3
    M0 open_w6_f0.75             95.35            95.8    0.03398     0.0937               0  3
    M0 open_w6_f0.90             114.4           114.5    0.05882     0.1887               0  3
    M0 open_w6_f1.10             139.8           127.4       2.11      3.191               1  3
    M1 open_w6_f0.25             22.15           23.27    0.03782    0.04526               0  3
    M1 open_w6_f0.50              44.3           44.77    0.03848    0.06264               0  3
    M1 open_w6_f0.75             66.45           67.53    0.04933     0.1421               0  3
    M1 open_w6_f0.90             79.74           81.27     0.0885     0.2519               0  3
    M1 open_w6_f1.10             97.46  

In [5]:
# RQ7: throughput against server workers W at C = 16, the speedup over W = 1, and per price pair the overhead
# (baseline over verifiable throughput) at each W. H7: within +-20 % across W when verification is client-side.
mc = cur[cur.experiment == "multicore"].sort_values(["arm", "workers"])
w1 = mc[mc.workers == 1].set_index("arm").throughput_qps
mc = mc.assign(speedup=mc.throughput_qps / mc.arm.map(w1))
print(mc[["arm", "workers", "throughput_qps", "speedup", "lat_p50_s", "server_cpu_util"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
cs = plots.core_scaling(cur)
print("\n", cs.pivot_table(index="pair", columns="workers", values="overhead").round(3).to_string())
spread = cs.groupby("pair").overhead.agg(lambda s: s.max() / s.min() - 1)
print("\nspread of the overhead across W (max / min - 1):", (100 * spread).round(1).to_dict())

   arm  workers  throughput_qps  speedup  lat_p50_s  server_cpu_util
    M0        1            40.2        1      0.398            200.5
    M0        2           79.07    1.967     0.2022            400.5
    M0        4           127.2    3.163     0.1248            750.9
    M0        6           127.4    3.168     0.1242              753
    M1        1           27.83        1     0.5752            200.3
    M1        2           54.63    1.963     0.2927            400.6
    M1        4           88.67    3.186     0.1795            753.3
    M1        6           88.87    3.193     0.1784              758
S0-cpp        1           66.67        1     0.2399            100.2
S0-cpp        2           131.7    1.976     0.1215            200.3
S0-cpp        4           261.3     3.92    0.06107            400.5
S0-cpp        6           378.8    5.682    0.04252            600.9
 S0-go        1           135.8        1     0.1178            101.6
 S0-go        2           270.7   

In [6]:
# F10 (latency against throughput) and F11 (throughput and overhead against W).
for name in ("F10_load_latency", "F11_cores"):
    save(name, plots.FIGURE_BUILDERS[name](t))
print(sorted(p.name for p in FIG.glob("F1[01]_*")))

['F10_load_latency.pdf', 'F10_load_latency.png', 'F11_cores.pdf', 'F11_cores.png']
